# AnomaliQ v3 — Experimento 01
**Detecção de DDoS com QSVM e VQC em simulador ideal**

Brazil Quantum Camp — 2026

## 1. Imports

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from preprocessing import preprocess
from feature_map import get_feature_map
from qsvm import train_qsvm, predict_qsvm
from vqc import train_vqc, predict_vqc
from alerts import generate_alerts

## 2. Dataset Simulado (placeholder para CICIDS2017)

In [ ]:
# Dataset sintético com 8 features (equivalente ao pipeline real)
X, y = make_classification(
    n_samples=200,
    n_features=8,
    n_informative=6,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

## 3. Pré-processamento

In [ ]:
N_QUBITS = 4

X_train_pca, X_test_pca, scaler, pca = preprocess(X_train, X_test, n_qubits=N_QUBITS)

print(f'Dimensão após PCA: {X_train_pca.shape}')

## 4. Quantum Feature Map

In [ ]:
feature_map = get_feature_map(n_qubits=N_QUBITS, reps=2, map_type='zz')
feature_map.decompose().draw('mpl')

## 5. Treinamento QSVM

In [ ]:
print('Treinando QSVM...')
qsvm_model = train_qsvm(feature_map, X_train_pca, y_train)

y_pred_qsvm = predict_qsvm(qsvm_model, X_test_pca)

print('\n--- QSVM ---')
print(classification_report(y_test, y_pred_qsvm))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_qsvm):.4f}')

## 6. Treinamento VQC

In [ ]:
print('Treinando VQC...')
vqc_model = train_vqc(feature_map, X_train_pca, y_train, n_qubits=N_QUBITS, maxiter=60)

y_pred_vqc = predict_vqc(vqc_model, X_test_pca)

print('\n--- VQC ---')
print(classification_report(y_test, y_pred_vqc))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_vqc):.4f}')

## 7. Geração de Alertas

In [ ]:
# Score simulado para demonstração
scores = np.random.uniform(0, 1, len(X_test_pca))
alertas = generate_alerts(scores, threshold=0.7)

print(f'Total de alertas gerados: {len(alertas)}')
print(f'Índices: {alertas}')

## 8. Comparativo de Resultados

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

resultados = pd.DataFrame({
    'Modelo': ['QSVM', 'VQC'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_qsvm),
        accuracy_score(y_test, y_pred_vqc)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_qsvm),
        f1_score(y_test, y_pred_vqc)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_qsvm),
        roc_auc_score(y_test, y_pred_vqc)
    ]
})

print(resultados.to_string(index=False))

resultados.set_index('Modelo').plot(kind='bar', figsize=(8,4), rot=0)
plt.title('AnomaliQ v3 — Comparativo QSVM vs VQC')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/exp01_comparativo.png', dpi=150)
plt.show()